In [2]:
# Importando bibliotecas do sistema para navegar nas pastas
import sys
import os

# Adicionando a pasta raiz do projeto (fincontrol) ao caminho do Python
sys.path.append(os.path.abspath('..'))

# Importando a nossa função criada no extractor.py
from src.extractor import extract_nubank_credit_card

# Definindo o caminho para o primeiro arquivo da sua lista
caminho_arquivo = '../data/bronze/credit_cards/nubank/Nubank_2017-02-11.ofx'

# Executando a extração
df_cartao = extract_nubank_credit_card(caminho_arquivo)

# Exibindo as primeiras linhas do resultado
display(df_cartao.head())

,id_transacao,data_compra,valor,descricao,tipo,data_fatura,banco
0,588070fa-9ce6-47b3-84b6-ebfda3da223d,2017-01-18 03:00:00,40.16,Pagamento recebido,credit,2017-01-25 03:00:00,NUBANK
1,5880724c-8d7c-41be-9134-a913f58e6e7d,2017-01-19 03:00:00,-36.75,Apl* Itunes.Com/Bill,debit,2017-01-25 03:00:00,NUBANK
2,5880724c-e926-4343-9fde-c090122b008e,2017-01-19 03:00:00,-2.34,"IOF de ""Apl* Itunes.Com/Bill""",debit,2017-01-25 03:00:00,NUBANK


In [3]:
import os
from src.transformer import clean_credit_card_data

# 1. Passando o dado bruto pela nossa função de limpeza
df_silver = clean_credit_card_data(df_cartao)

# 2. Criando o caminho da pasta Silver (garantindo que ela exista na nossa estrutura)
caminho_silver = '../data/silver/credit_cards/nubank/'
os.makedirs(caminho_silver, exist_ok=True)

# 3. Definindo o nome do arquivo de saída e salvando em formato Parquet
arquivo_saida = os.path.join(caminho_silver, 'Nubank_2017-02-11.parquet')
df_silver.to_parquet(arquivo_saida, index=False)

print(f"Arquivo salvo com sucesso em: {arquivo_saida}")

# Exibindo o resultado final limpo
display(df_silver.head())

Arquivo salvo com sucesso em: ../data/silver/credit_cards/nubank/Nubank_2017-02-11.parquet


,id_transacao,data_compra,valor,descricao,tipo,data_fatura,banco
0,588070fa-9ce6-47b3-84b6-ebfda3da223d,2017-01-18 03:00:00,40.16,PAGAMENTO RECEBIDO,credit,2017-01-25 03:00:00,NUBANK
1,5880724c-8d7c-41be-9134-a913f58e6e7d,2017-01-19 03:00:00,-36.75,APL* ITUNES.COM/BILL,debit,2017-01-25 03:00:00,NUBANK
2,5880724c-e926-4343-9fde-c090122b008e,2017-01-19 03:00:00,-2.34,"IOF DE ""APL* ITUNES.COM/BILL""",debit,2017-01-25 03:00:00,NUBANK


In [4]:
import os
import glob
import pandas as pd
from src.extractor import extract_nubank_credit_card
from src.transformer import clean_credit_card_data

# 1. Definir a pasta onde estão os arquivos brutos do Nubank
caminho_bronze_nubank = '../data/bronze/credit_cards/nubank/'

# O glob vai listar todos os arquivos terminados em .ofx dentro dessa pasta
arquivos_ofx = glob.glob(os.path.join(caminho_bronze_nubank, '*.ofx'))
print(f"Encontrados {len(arquivos_ofx)} arquivos OFX para processar. Iniciando...\n")

# 2. Lista vazia que vai guardar os DataFrames de cada arquivo
lista_dfs = []

# 3. O Loop da automação
for arquivo in arquivos_ofx:
    try:
        df_bruto = extract_nubank_credit_card(arquivo)
        df_limpo = clean_credit_card_data(df_bruto)
        lista_dfs.append(df_limpo)
    except Exception as e:
        print(f"⚠️ Erro ao processar o arquivo {os.path.basename(arquivo)}: {e}")

# 4. Juntando as peças e finalizando
if lista_dfs:
    df_consolidado = pd.concat(lista_dfs, ignore_index=True)
    
    # REGRA ATUALIZADA: Desduplicação inteligente!
    # Só deleta se a transação tiver o mesmo ID E estiver na mesma fatura.
    # Isso preserva as parcelas futuras que você já baixou.
    qtd_antes = len(df_consolidado)
    df_consolidado = df_consolidado.drop_duplicates(subset=['id_transacao', 'data_fatura'])
    qtd_depois = len(df_consolidado)
    
    # Ordena para o histórico ficar visualmente coerente
    df_consolidado = df_consolidado.sort_values(by=['data_fatura', 'data_compra']).reset_index(drop=True)
    
    # 5. Salva o histórico final na camada Silver
    caminho_silver = '../data/silver/credit_cards/nubank/'
    os.makedirs(caminho_silver, exist_ok=True) # Garante que a pasta existe
    
    caminho_silver_consolidado = os.path.join(caminho_silver, 'nubank_historico_consolidado.parquet')
    df_consolidado.to_parquet(caminho_silver_consolidado, index=False)
    
    print(f"✅ Sucesso! O histórico final foi salvo.")
    print(f"   - Transações totais processadas: {qtd_antes}")
    print(f"   - Transações únicas consolidadas: {qtd_depois}")
    print(f"   - Foram removidas {qtd_antes - qtd_depois} compras duplicadas (mantendo as parcelas futuras seguras!).\n")
    
    display(df_consolidado.head())
else:
    print("Nenhum dado foi consolidado.")

Encontrados 112 arquivos OFX para processar. Iniciando...

✅ Sucesso! O histórico final foi salvo.
   - Transações totais processadas: 6579
   - Transações únicas consolidadas: 6538
   - Foram removidas 41 compras duplicadas (mantendo as parcelas futuras seguras!).



,id_transacao,data_compra,valor,descricao,tipo,data_fatura,banco
0,588070fa-9ce6-47b3-84b6-ebfda3da223d,2017-01-18 03:00:00,40.16,PAGAMENTO RECEBIDO,credit,2017-01-25 03:00:00,NUBANK
1,5880724c-8d7c-41be-9134-a913f58e6e7d,2017-01-19 03:00:00,-36.75,APL* ITUNES.COM/BILL,debit,2017-01-25 03:00:00,NUBANK
2,5880724c-e926-4343-9fde-c090122b008e,2017-01-19 03:00:00,-2.34,"IOF DE ""APL* ITUNES.COM/BILL""",debit,2017-01-25 03:00:00,NUBANK
3,588b071b-e876-4e70-a174-68dfbac60aa6,2017-01-27 03:00:00,-28.24,CAFFE RESTAURANTES,debit,2017-02-22 03:00:00,NUBANK
4,588b7523-52da-481d-afe8-422b33595e66,2017-01-27 03:00:00,-27.01,AUTO POSTO AMAZONAS,debit,2017-02-22 03:00:00,NUBANK


In [5]:
# Importando a nova função do BB (certifique-se de que a clean_credit_card_data já está importada)
from src.extractor import extract_bb_credit_card

# 1. Definir a pasta onde estão os arquivos brutos do BB
caminho_bronze_bb = '../data/bronze/credit_cards/banco_do_brasil/'

# Listando os arquivos .ofx do BB (ignorando os PDFs e TXTs por enquanto)
arquivos_ofx_bb = glob.glob(os.path.join(caminho_bronze_bb, '*.ofx'))
print(f"Encontrados {len(arquivos_ofx_bb)} arquivos OFX do BB para processar. Iniciando...\n")

lista_dfs_bb = []

# 2. O Loop da automação para o BB
for arquivo in arquivos_ofx_bb:
    try:
        df_bruto_bb = extract_bb_credit_card(arquivo)
        df_limpo_bb = clean_credit_card_data(df_bruto_bb)
        lista_dfs_bb.append(df_limpo_bb)
    except Exception as e:
        print(f"⚠️ Erro ao processar o arquivo {os.path.basename(arquivo)}: {e}")

# 3. Consolidando e salvando
if lista_dfs_bb:
    df_consolidado_bb = pd.concat(lista_dfs_bb, ignore_index=True)
    
    qtd_antes_bb = len(df_consolidado_bb)
    df_consolidado_bb = df_consolidado_bb.drop_duplicates(subset=['id_transacao', 'data_fatura'])
    qtd_depois_bb = len(df_consolidado_bb)
    
    # Se alguma data_fatura veio vazia (None), vamos preencher temporariamente com a data da compra
    # para não quebrar a ordenação
    df_consolidado_bb['data_fatura'] = df_consolidado_bb['data_fatura'].fillna(df_consolidado_bb['data_compra'])
    
    df_consolidado_bb = df_consolidado_bb.sort_values(by=['data_fatura', 'data_compra']).reset_index(drop=True)
    
    # 4. Salva o histórico final do BB na camada Silver
    caminho_silver_bb = '../data/silver/credit_cards/banco_do_brasil/'
    os.makedirs(caminho_silver_bb, exist_ok=True)
    
    caminho_silver_consolidado_bb = os.path.join(caminho_silver_bb, 'bb_historico_consolidado.parquet')
    df_consolidado_bb.to_parquet(caminho_silver_consolidado_bb, index=False)
    
    print(f"✅ Sucesso! O histórico do BB foi salvo.")
    print(f"   - Transações processadas: {qtd_antes_bb}")
    print(f"   - Transações únicas consolidadas: {qtd_depois_bb}\n")
    
    display(df_consolidado_bb.head())
else:
    print("Nenhum dado do BB foi consolidado.")

Encontrados 69 arquivos OFX do BB para processar. Iniciando...

✅ Sucesso! O histórico do BB foi salvo.
   - Transações processadas: 2720
   - Transações únicas consolidadas: 2720



,id_transacao,data_compra,valor,descricao,tipo,data_fatura,banco
0,2020022649840000000009750000000017,2020-02-26,-205.99,AZUL *UB9NR PARC 03/06 BARUERI BR,payment,2020-05-19,BB
1,2020030549840000000009750000000018,2020-03-05,-114.95,LOJAS MILLA PARC 03/04 PORTO VELHO BR,payment,2020-05-19,BB
2,2020031149840000000009750000000020,2020-03-11,-153.64,AZUL *XCL48 PARC 03/06 BARUERI BR,payment,2020-05-19,BB
3,2020041649840000000009750000000003,2020-04-16,-28.00,ELIZABETE FERREIRA DE PORTO VELHO BR,payment,2020-05-19,BB
4,2020041749840000000009750000000007,2020-04-17,-210.83,POSTO MIRIAN PORTO VELHO BR,payment,2020-05-19,BB
